In [ ]:
#Compruebo libreria request instalada
#Necesaría para hacer solicitudes API
!pip install requests

In [ ]:
import requests #Para poder hacer solicitudes a la API
import pandas as pd
import json
import re
import time
import io
import csv
import urllib.request
from collections import Counter
import xml.etree.ElementTree as ET
from pathlib import Path
from google.colab import drive, files
import shutil # copiar docs. drive en disco local
pd.options.mode.copy_on_write = True

In [ ]:
# Montar drive:
#drive.mount('/content/drive')

# DATASETS DISPONIBLES

En datos abiertos Ayuntamiento Madrid

In [ ]:
# Datasets de los datos abiertos
url = "https://datos.madrid.es/api/3/action/package_list"
r = requests.get(url, timeout=10)

print(r.status_code, r.headers.get("Content-Type"))
r.raise_for_status()  #Comprueba el código de estado HTTP de la respuesta

data = r.json()
print(data["success"]) # True si ha ido bien
print(f"Nº Datasets encontrados en datos abiertos madrid: {len(data["result"])}")

In [ ]:
#Se busca codigo dataset biblioteca publica:
resultados_consulta = data['result']
dtasets_bibliotecas = []
valor_dataset = "bibliotecas"

for i in resultados_consulta:
  if valor_dataset.lower() in i.lower():
    dtasets_bibliotecas.append(i)

print(f"{len(dtasets_bibliotecas)} Datasets encontrados en datos abiertos "
f"madrid sobre las bibliotecas:\n{dtasets_bibliotecas}")

In [ ]:
#Se busca codigo dataset poblacion publica:
resultados_consulta = data['result']
dtasets_poblacion  = []
valor_dataset = "poblacion"

for i in resultados_consulta:
  if valor_dataset.lower() in i.lower():
    dtasets_poblacion .append(i)

print(f"{len(dtasets_poblacion )} Datasets encontrados en datos abiertos "
f"madrid sobre las poblacion :\n{dtasets_poblacion}")

## Prestamo

In [ ]:
# ---------------- Configuración ----------------
DATASET_ID = "212700-0-bibliotecas-prestamos-historico"
API = f"https://datos.madrid.es/api/3/action/package_show?id={DATASET_ID}"
ANIO_INICIO = 2018
ANIO_FIN = 2026
DESTINO = Path("") # Guardar descargas
PREFIJO_NUMERICO = True   # True -> 2025_12_diciembre.csv (ordena cronológicamente)
PAUSA_SEG = 1


MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio",
         "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
PATRON_TITULO = re.compile(r"Préstamos\s+(\w+)\s+(\d{4})", re.IGNORECASE)

sesion = requests.Session()
sesion.headers["User-Agent"] = "Mozilla/5.0 (descarga datos abiertos)"


def obtener_recursos():
    """Devuelve {(año, mes): url_csv} a partir de la API CKAN."""
    r = sesion.get(API, timeout=60)
    r.raise_for_status()
    datos = json.loads(r.text)  # el servidor etiqueta la respuesta como text/html, pero es JSON

    if not datos.get("success"):
        raise RuntimeError(f"La API devolvió un error: {datos.get('error')}")

    recursos = {}
    for res in datos["result"]["resources"]:
        if (res.get("format") or "").upper() != "CSV":
            continue
        anio_mes = año_mes_del_recurso(res)
        if anio_mes is None:
            print(f"  (sin año/mes reconocible, se omite: {res.get('id')})")
            continue
        anio, mes = anio_mes
        if ANIO_INICIO <= anio <= ANIO_FIN:
            recursos.setdefault((anio, mes), res["url"])
    return recursos


def año_mes_del_recurso(res):
    """
    Saca (año, mes) del campo 'hierarchy'; si no, del texto de 'description'.
    """
    nombres = [h.get("hierarchy_name", "").strip().lower() for h in res.get("hierarchy") or []]
    anios = [int(n) for n in nombres if n.isdigit() and len(n) == 4]
    meses = [n for n in nombres if n in MESES]
    if anios and meses:
        return anios[0], meses[0]

    m = PATRON_TITULO.search(res.get("description") or "")
    if m and m.group(1).lower() in MESES:
        return int(m.group(2)), m.group(1).lower()
    return None


def nombre_archivo(anio, mes):
    if PREFIJO_NUMERICO:
        return f"{anio}_{MESES.index(mes) + 1:02d}_{mes}.csv"
    return f"{anio}_{mes}.csv"


def descargar(url, ruta):
    tmp = ruta.with_suffix(".part")
    with sesion.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for bloque in r.iter_content(chunk_size=1024 * 1024):
                f.write(bloque)
    tmp.rename(ruta)


def descargar_prestamos():
    DESTINO.mkdir(parents=True, exist_ok=True)
    print("Guardando en:", DESTINO.resolve())

    recursos = obtener_recursos()
    print(f"Encontrados {len(recursos)} CSV entre {ANIO_INICIO} y {ANIO_FIN}.\n")

    errores = []
    for (anio, mes) in sorted(recursos, key=lambda k: (k[0], MESES.index(k[1]))):
        ruta = DESTINO / nombre_archivo(anio, mes)
        if ruta.exists():
            print(f"  ya existe   {ruta.name}")
            continue
        try:
            print(f"  descargando {ruta.name} ...", end=" ", flush=True)
            descargar(recursos[(anio, mes)], ruta)
            print(f"OK ({ruta.stat().st_size / 1e6:.1f} MB)")
        except Exception as e:
            print(f"ERROR: {e}")
            errores.append(ruta.name)
        time.sleep(PAUSA_SEG)

    faltan = [f"{a}_{m}" for a in range(ANIO_INICIO, ANIO_FIN + 1)
              for m in MESES if (a, m) not in recursos]
    print("\nMeses sin CSV publicado:", ", ".join(faltan) or "ninguno")
    if errores:
        print("Fallaron (vuelve a ejecutar para reintentar):", ", ".join(errores))


## Poblacion por distrito y barrio

In [ ]:
# Variables:
ORIGEN = Path("")
DESTINO = Path("") # Guardar descargas
ORIGEN.mkdir(parents=True, exist_ok=True)
DESTINO.mkdir(parents=True, exist_ok=True)

URL_PADRON = "https://datos.madrid.es/egob/catalogo/300557-0-poblacion-distrito-barrio.csv"

# Funciones:
def decodifica(contenido: bytes) -> str:
    """Prueba utf-8 y, si falla, latin-1 (habitual en ficheros públicos españoles)."""
    try:
        return contenido.decode("utf-8-sig")
    except UnicodeDecodeError:
        return contenido.decode("latin-1")

def descargar_poblacion():
    r = requests.get(URL_PADRON, timeout=120)
    r.raise_for_status()
    (ORIGEN / "300557_poblacion_distrito_barrio.csv").write_bytes(r.content)

    pob = pd.read_csv(io.StringIO(decodifica(r.content)), sep=None, engine="python",
                     thousands=".", decimal=",")
    pob.to_csv(DESTINO/'poblacion_2018_2026.csv', index=False, encoding="utf-8")


# Creacion CSV códigos descriptivos tipo publico destinatario prcoc1

Creo DF con los datos de los códigos prcoc1 ya que no está disponible el CSV en la documentación oficial

Extraigo códigos y descripción de ellos del documento adjunto de la BBDD '207075-0-bibliotecas-prestamos', que corresponde a los prestamos activos de la red de Bibliotecas Publicas de Madrid. La tabla tiene la misma estructura que las presentes en la BBDD '212700-0-bibliotecas-prestamos-historico'.

El resto de tablas soporte códigos descritos descargados manualmente.


In [ ]:
# Datos del DF:
datos_prcoc1 = {
    'Codigo': ['ADU', 'INF', 'JUV', 'OTC', 'PLE', 'XXX'],
    'Descripcion': ['Adultos',
                    'Infantil',
                    'Juvenil',
                    'Otros códigos de ejemplar',
                    'Prelectores',
                    'Mayores de 18 años']
    }

# Creación del DF:
df_prcoc1 = pd.DataFrame(datos_prcoc1).astype({'codigo':'str', 'descripcion':'str'})

# Guardado CSV:
df_prcoc1.to_csv(
    carpeta_final/ 'codigo_lector_destino.csv',
    index=False,
    sep= ";",
    quotechar= '"',
    encoding='utf-8-sig' # Codificacion que tendran todos los CSV al final
    )

# Creacion CSVs ejemplares y registros provenientes del catalogoXML

## Registros y ejemplares

In [ ]:
# Extraer registros y ejemplares del catalogo

"""
extraer_catalogo_marc_v4.py
Capa: bronze -> silver (extracción fiel del MARCXML, no decide)

Lee el catálogo MARC 21 (MARCXML, un único XML de ~3,2 GB) en streaming
(iterparse) y genera:
  - registros_v4.csv   -> 1 fila por registro bibliográfico (campo 001)
  - ejemplares_v4.csv  -> 1 fila por ejemplar (campo 852)
  - resumen_extraccion_v4.txt -> cobertura (% no vacío) de cada columna nueva
                                 y prefijos de la CDU (para detectar notación antigua)
"""
# Contantes


RUTA_XML = Path("/content/catalogo.xml") # copia local del catalogoXML
CARPETA_SALIDA = Path("drive")
LIMITE = None  # prueba; None = fichero completo
AVISO_CADA = 100_000   # frecuencia del aviso de progreso
SEP = "|"              # separador de valores repetibles

# ---------------------------------------------------------------------------
# COLUMNAS DE SALIDA (el orden define el CSV)
# ---------------------------------------------------------------------------
COLS_EJEMPLARES = [
    "barcode_norm", "barcode_original", "id_registro",
    "biblioteca_852a", "sucursal_852b", "localizacion_852c",
    "signatura_852j", "subcampo_852m",
]

COLS_REGISTROS = [
    "id_registro",
    # --- Leader ---
    "ld06", "ld07", "es_libro",
    # --- 008 (posiciones fijas) ---
    "len_008",            # necesario en para anular 22/33 si el 008 está desalineado
    "tipo_fecha_008",     # 008/06: explica qué significan fecha1 y fecha2
    "fecha1_008",         # 008/07-10
    "fecha2_008",         # 008/11-14 (con tipo "r" = fecha de la edición original)
    "pais_edicion_008",   # 008/15-17
    "lengua_texto_008",   # 008/35-37
    "publico_008",        # 008/22 (solo válido para libros)
    "forma_literaria_008",  # 008/33 (solo válido para libros)
    # --- Lengua y traducción ---
    "traduccion_041_ind1",  # 041 ind1: 1 = es o incluye traducción
    "lengua_texto_041a",    # 041 $a
    "lengua_original_041h", # 041 $h  <- lengua original
    "lengua_intermedia_041k",  # 041 $k (traducciones indirectas)
    "traductor_700",        # 1 si algún 700/710 tiene función de traductor
    "mencion_traduccion_245c",  # 1 si la mención de responsabilidad cita una traducción
    # --- Títulos ---
    "titulo_245a",
    "titulo_orden_245a",    # 245 $a sin artículo inicial (según 245 ind2)
    "subtitulo_245b",
    "parte_245np",          # 245 $n $p (número/nombre de parte: tomos, sagas)
    "titulo_uniforme_240",  # 240 $a $n $p
    "lengua_240l",          # 240 $l (lengua de la versión, p. ej. "Español")
    "titulo_uniforme_130",  # 130 $a $n $p
    "titulo_original_nota", # "Tít. orig." en 500 $a o 246 con $i "original"
    # --- Responsabilidad y publicación ---
    "autor",                # 100/110/111 $a
    "isbn_original",        # 020 $a (todos), sin normalizar
    "anio_publicacion",     # año de 260 $c (o 264 ind2=1 $c)
    # --- Clasificación y materias ---
    "cdu_080",              # 080 $a (todos)
    "forma_literaria_cdu",  # auxiliar de forma, nivel 1 (p. ej. "3", "93")
    "genero_literario_cdu", # auxiliar de forma completo (p. ej. "31", "311.6")
    "lengua_literatura_cdu",  # 821.xxx -> "134.2"
    "lugar_literatura_cdu",   # auxiliar de lugar tras la lengua, p. ej. "7/8"
    "critica_literaria_cdu",  # 1 si hay 82...09 (crítica/estudios: no ficción)
    "materias_650",
    "generos_655",
    # --- Público (521) ---
    "tipo_publico_521",       # 521 ind1
    "publico_destinatario_521a",
    "nota_publico_521",       # todos los subcampos de cada 521
    # --- Ejemplares ---
    "n_ejemplares",
]

# ---------------------------------------------------------------------------
# FUNCIONES AUXILIARES
# ---------------------------------------------------------------------------
def tag_local(tag):
    """Quita el namespace si existiera ('{ns}record' -> 'record')."""
    return tag.rsplit("}", 1)[-1]


def limpiar(texto) -> str:
    """Colapsa espacios y quita la puntuación ISBD de los extremos
    (' : Espasa-Calpe' -> 'Espasa-Calpe', '-S.XVI' -> 'S.XVI', ', 1979' -> '1979')."""
    if not texto:
        return ""
    texto = " ".join(texto.split())
    texto = re.sub(r"^[\s:;,./=\-]+|[\s:;,/=]+$", "", texto)
    # Paréntesis sueltos: MARC parte "(1948-2014)" entre $a y $d, y deja
    # 'Panero, Leopoldo María (' y '1948-2014)'. Se quita el '(' final y los ')' sin pareja.
    texto = re.sub(r"\s*\($", "", texto)
    while texto.endswith(")") and texto.count(")") > texto.count("("):
        texto = texto[:-1].rstrip(" :;,/=")
    return texto.replace(SEP, "/")


def normalizar_barcode(texto) -> str:
    """' 101482046' / '0101482046' -> '101482046'.
    Aplicar la MISMA función a prbarc."""
    digitos = re.sub(r"\D", "", texto or "")
    return str(int(digitos)) if digitos else ""


def extraer_anio(texto) -> str:
    """Primer año plausible (1500-2099) de cadenas como 'cop. 2010', '[2005?]', ', 1979'."""
    m = re.search(r"(1[5-9]\d{2}|20\d{2})", texto or "")
    return m.group(1) if m else ""


def pos(cadena, ini, fin=None):
    """Extrae posiciones fijas; '' si la cadena es corta. Blanco -> '#'."""
    fin = ini + 1 if fin is None else fin
    if len(cadena) < fin:
        return ""
    return cadena[ini:fin].replace(" ", "#")


def unir(valores):
    """Une valores no vacíos sin duplicados, respetando el orden."""
    vistos = []
    for v in valores:
        if v and v not in vistos:
            vistos.append(v)
    return SEP.join(vistos)


# --- CDU: desglose de la literatura (clase 82) -----------------------------
# Notación típica: 821.134.2(7/8)-31"19"  -> lengua 134.2, lugar 7/8, forma 31
RE_LIT = re.compile(r"^\s*82(?P<resto>.*)$")
RE_LENGUA = re.compile(r"^1\.(?P<lengua>\d+(?:\.[1-9]\d*)*)")  # excluye .0x (p. ej. .09 crítica)
RE_LUGAR = re.compile(r"^\((?P<lugar>[^)]*)\)")
RE_FORMA = re.compile(r"-(?P<forma>\d[\d.]*)")


def desglosar_cdu_literaria(valor):
    """
    Devuelve (forma_nivel1, forma_completa, lengua, lugar, es_critica)
    para una notación de la clase 82. Si no es 82..., devuelve todo vacío.
    - forma: auxiliar especial de forma literaria (82-1 poesía, -2 teatro, -3 prosa...)
      Nivel 1 = 1 dígito, salvo -8x y -9x, donde la clase significativa tiene 2.
    - critica: '.09' tras la notación de literatura = crítica/estudios literarios.
    """
    m = RE_LIT.match(valor)
    if not m:
        return "", "", "", "", ""
    resto = m.group("resto")
    lengua = lugar = ""
    ml = RE_LENGUA.match(resto)     # 821.134.2 -> resto empieza por "1.134.2"
    if ml:
        lengua = ml.group("lengua")
        tras = resto[ml.end():]
        mp = RE_LUGAR.match(tras)
        if mp:
            lugar = mp.group("lugar")
    mf = RE_FORMA.search(resto)
    forma_completa = mf.group("forma").rstrip(".") if mf else ""
    if forma_completa:
        forma_n1 = forma_completa[:2] if forma_completa[0] in "89" else forma_completa[0]
    else:
        forma_n1 = ""
    critica = "1" if re.search(r"\.09(?!\d)", resto) else ""
    return forma_n1, forma_completa, lengua, lugar, critica


RE_TIT_ORIG = re.compile(r"t[íi]t(?:\.|ulo)\s*orig(?:\.|inal)?\s*[:.]?\s*(.+)", re.I)
RE_TRAD = re.compile(r"\btradu", re.I)


def es_traductor(subcampos):
    """700/710: $4 'trl' o $e que empiece por 'tr' (tr., trad., traductor)."""
    for code, txt in subcampos:
        t = txt.lower().strip()
        if code == "4" and t.startswith("trl"):
            return True
        if code == "e" and (t.startswith("tr") or "tradu" in t):
            return True
    return False


# ---------------------------------------------------------------------------
# PROCESADO DE UN REGISTRO
# ---------------------------------------------------------------------------
def procesar_registro(rec):
    """Convierte un <record> en (fila_registro, [filas_ejemplar])."""
    leader = ""
    control = {}
    campos = []   # lista de (tag, ind1, ind2, [(code, texto), ...])

    for hijo in rec:
        t = tag_local(hijo.tag)
        if t == "leader":
            leader = hijo.text or ""
        elif t == "controlfield":
            control.setdefault(hijo.get("tag"), hijo.text or "")
        elif t == "datafield":
            subs = [(s.get("code"), limpiar(s.text)) for s in hijo
                    if tag_local(s.tag) == "subfield"]
            campos.append((hijo.get("tag"), hijo.get("ind1", " "),
                           hijo.get("ind2", " "), subs))

    def campos_tag(tag):
        return [c for c in campos if c[0] == tag]

    def subs_de(campo, codigos):
        return [txt for code, txt in campo[3] if code in codigos and txt]

    id_reg = limpiar(control.get("001", ""))
    c008 = control.get("008", "")

    ld06, ld07 = pos(leader, 6), pos(leader, 7)
    fila = {c: "" for c in COLS_REGISTROS}
    fila.update({
        "id_registro": id_reg,
        "ld06": ld06,
        "ld07": ld07,
        "es_libro": "1" if ld06 == "a" and ld07 in ("m", "c") else "0",
        "len_008": str(len(c008)),
        "tipo_fecha_008": pos(c008, 6),
        "fecha1_008": pos(c008, 7, 11),
        "fecha2_008": pos(c008, 11, 15),
        "pais_edicion_008": pos(c008, 15, 18).rstrip("#"),  # códigos de 2 letras llevan blanco
        "publico_008": pos(c008, 22),
        "forma_literaria_008": pos(c008, 33),
        "lengua_texto_008": pos(c008, 35, 38),
    })

    # --- 041 lengua / traducción ---
    ind1_041, a041, h041, k041 = [], [], [], []
    for c in campos_tag("041"):
        ind1_041.append(c[1].replace(" ", "#"))
        a041 += subs_de(c, "a")
        h041 += subs_de(c, "h")
        k041 += subs_de(c, "k")
    fila["traduccion_041_ind1"] = unir(ind1_041)
    fila["lengua_texto_041a"] = unir(a041)
    fila["lengua_original_041h"] = unir(h041)
    fila["lengua_intermedia_041k"] = unir(k041)
    fila["traductor_700"] = "1" if any(es_traductor(c[3]) for c in campos_tag("700") + campos_tag("710")) else "0"

    # --- 245 título ---
    c245 = campos_tag("245")
    if c245:
        c = c245[0]
        tit = limpiar(" ".join(subs_de(c, "a")))
        fila["titulo_245a"] = tit
        nf = int(c[2]) if c[2].isdigit() else 0          # ind2 = caracteres no alfabetizables
        fila["titulo_orden_245a"] = tit[nf:].strip() if nf else tit
        fila["subtitulo_245b"] = limpiar(" ".join(subs_de(c, "b")))
        fila["parte_245np"] = limpiar(" ".join(subs_de(c, "np")))
        fila["mencion_traduccion_245c"] = "1" if RE_TRAD.search(" ".join(subs_de(c, "c"))) else "0"

    # --- títulos uniformes ---
    for tag, col in (("240", "titulo_uniforme_240"), ("130", "titulo_uniforme_130")):
        cs = campos_tag(tag)
        if cs:
            fila[col] = limpiar(" ".join(subs_de(cs[0], "anp")))
    c240 = campos_tag("240")
    if c240:
        fila["lengua_240l"] = limpiar(" ".join(subs_de(c240[0], "l")))

    # --- título original en notas (500) o variante (246 $i "...original...") ---
    orig = []
    for c in campos_tag("500"):
        for txt in subs_de(c, "a"):
            m = RE_TIT_ORIG.search(txt)
            if m:
                orig.append(limpiar(m.group(1)))
    for c in campos_tag("246"):
        if any("orig" in t.lower() for t in subs_de(c, "i")):
            orig.append(limpiar(" ".join(subs_de(c, "a"))))
    fila["titulo_original_nota"] = unir(orig)

    # --- autor ---
    # Primer 100/110/111 en orden de registro.
    for c in campos:
        if c[0] in ("100", "110", "111"):
            fila["autor"] = (subs_de(c, "a") or [""])[0]
            break

    # --- ISBN ---
    fila["isbn_original"] = unir(t for c in campos_tag("020") for t in subs_de(c, "a"))

    # --- año de publicación ---
    # primer 260; si no hay, primer 264 con ind2=1 (publicación).
    pub = (campos_tag("260") or [None])[0]
    if pub is None:
        pub = next((c for c in campos_tag("264") if c[2] == "1"), None)
    if pub is not None:
        fila["anio_publicacion"] = extraer_anio(" ".join(subs_de(pub, "c")))

    # --- CDU y desglose literario ---
    cdus = [t for c in campos_tag("080") for t in subs_de(c, "a")]
    fila["cdu_080"] = unir(cdus)
    formas, generos, lenguas, lugares, criticas = [], [], [], [], []
    for v in cdus:
        f1, fc, lg, lu, cr = desglosar_cdu_literaria(v)
        formas.append(f1); generos.append(fc); lenguas.append(lg)
        lugares.append(lu); criticas.append(cr)
    fila["forma_literaria_cdu"] = unir(formas)
    fila["genero_literario_cdu"] = unir(generos)
    fila["lengua_literatura_cdu"] = unir(lenguas)
    fila["lugar_literatura_cdu"] = unir(lugares)
    fila["critica_literaria_cdu"] = "1" if "1" in criticas else ""

    # --- materias y géneros ---
    # 650: término + subdivisiones ($x general, $y cronológica, $z geográfica,
    # $j forma) unidas con ' -- '
    fila["materias_650"] = unir(" -- ".join(subs_de(c, "axyzj")) for c in campos_tag("650"))
    fila["generos_655"] = unir(limpiar(t) for c in campos_tag("655") for t in subs_de(c, "a"))

    # --- 521 público ---
    c521 = campos_tag("521")
    fila["tipo_publico_521"] = unir(c[1].replace(" ", "#") for c in c521)
    fila["publico_destinatario_521a"] = unir(t for c in c521 for t in subs_de(c, "a"))
    fila["nota_publico_521"] = unir(" ".join(t for _, t in c[3] if t) for c in c521)

    # --- ejemplares (852) ---
    ejemplares = []
    c852 = campos_tag("852")
    fila["n_ejemplares"] = str(len(c852))
    for c in c852:
        def primero(code):
            v = subs_de(c, code)
            return v[0] if v else ""
        bc = primero("p")
        ejemplares.append({
            "barcode_norm": normalizar_barcode(bc),
            "barcode_original": bc,
            "id_registro": id_reg,
            "biblioteca_852a": primero("a"),
            "sucursal_852b": primero("b"),
            "localizacion_852c": primero("c"),
            "signatura_852j": primero("j"),
            "subcampo_852m": primero("m"),
        })
    return fila, ejemplares


# ---------------------------------------------------------------------------
# PROGRAMA PRINCIPAL
# ---------------------------------------------------------------------------
def lectura_catalogo_xml():
    CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
    tam_total = RUTA_XML.stat().st_size
    no_vacios = Counter()          # cobertura por columna
    prefijos_cdu = Counter()       # dos primeros caracteres de cada 080 $a
    n_reg = n_ej = 0
    t0 = time.time()

    with open(RUTA_XML, "rb") as fxml, \
         open(CARPETA_SALIDA / "registros_v4.csv", "w", newline="", encoding="utf-8") as freg, \
         open(CARPETA_SALIDA / "ejemplares_v4.csv", "w", newline="", encoding="utf-8") as fej:

        w_reg = csv.DictWriter(freg, fieldnames=COLS_REGISTROS)
        w_ej = csv.DictWriter(fej, fieldnames=COLS_EJEMPLARES)
        w_reg.writeheader()
        w_ej.writeheader()

        contexto = ET.iterparse(fxml, events=("start", "end"))
        _, raiz = next(contexto)          # primer 'start' = elemento raíz

        for evento, elem in contexto:
            if evento != "end" or tag_local(elem.tag) != "record":
                continue
            fila, ejemplares = procesar_registro(elem)
            w_reg.writerow(fila)
            w_ej.writerows(ejemplares)
            n_reg += 1
            n_ej += len(ejemplares)
            for k, v in fila.items():
                # No cuentan como informados: vacío, '0' (indicador desactivado) ni los
                # códigos formados solo por '#' (blanco) o '|' (no codificado) del leader/008.
                # Ojo: en forma_literaria_008 el '0' (no ficción) SÍ es un valor informado.
                if k == "forma_literaria_008":
                    informado = v not in ("", "#", "|")
                else:
                    informado = v not in ("", "0") and v.strip("#|") != ""
                if informado:
                    no_vacios[k] += 1
            for v in fila["cdu_080"].split(SEP):
                if v:
                    prefijos_cdu[v.strip()[:2]] += 1

            raiz.clear()                  # libera memoria de los registros ya leídos

            if n_reg % AVISO_CADA == 0:
                pct = 100 * fxml.tell() / tam_total
                print(f"{n_reg:,} registros | {pct:5.1f} % del fichero | {time.time()-t0:,.0f} s")
            if LIMITE and n_reg >= LIMITE:
                break
        pct_leido = 100 * fxml.tell() / tam_total   # 100 % si se leyó entero

    # --- Resumen de cobertura ---
    lineas = [
        f"Registros: {n_reg:,} | Ejemplares: {n_ej:,} | Tiempo: {time.time()-t0:,.0f} s",
        f"Fichero leído: {pct_leido:.1f} % de {tam_total:,} bytes",
        "",
        "Cobertura por columna (registros con valor distinto de vacío y de '0'):",
    ]
    for c in COLS_REGISTROS:
        lineas.append(f"  {c:28s} {no_vacios[c]:>9,}  ({100*no_vacios[c]/max(n_reg,1):5.1f} %)")
    lineas += ["", "Prefijos de la CDU (2 primeros caracteres de cada 080 $a), top 30:"]
    for p, n in prefijos_cdu.most_common(30):
        lineas.append(f"  {p!r:6s} {n:>9,}")
    texto = "\n".join(lineas)
    (CARPETA_SALIDA / "resumen_extraccion_v4.txt").write_text(texto, encoding="utf-8")
    print(texto)


lectura_catalogo_xml()

## Tablas soporte

Fuentes:
  - Leader/06, Leader/07, 008/06, 008/22, 008/33, 041 ind1, 521 ind1:
    MARC 21 Format for Bibliographic Data (Library of Congress). Traducción propia.
  - Países (008/15-17) y lenguas (008/35-37, 041 $a/$h/$k): listas oficiales
    MARC Code List for Countries / for Languages, descargadas en XML de loc.gov.
    Se descargan en inglés; la traducción al español se hará solo para los
    códigos que aparezcan con peso en los datos.
  - CDU (clase 82): auxiliares de forma y lenguas de la literatura.
    La columna "verificado" indica si la equivalencia se ha comprobado en una
    fuente (ver columna "fuente"); las que dicen "no" hay que confirmarlas
    antes de usarlas en el informe.

In [ ]:
# Tablas CSV (codigo -> descripción) por cada columna codificada



CARPETA_SALIDA = Path("/content/codigos_marc2")   # luego -> 02_silver/catalogo/codigos
# Si loc.gov rechaza la descarga, baja los dos XML con el navegador, guárdalos en esta
# carpeta (sin cambiarles el nombre) y vuelve a ejecutar: el script los leerá de aquí.
CARPETA_XML_LOC = Path("/content")


LOC = "Library of Congress, MARC 21 Format for Bibliographic Data"

# ---------------------------------------------------------------------------
# 1. TABLAS FIJAS DEL ESTÁNDAR MARC 21 (el blanco se escribe "#", como en el CSV)
# ---------------------------------------------------------------------------
TABLAS_FIJAS = {
    "cod_leader06_tipo_registro": [
        ("a", "Material textual (impreso)"), ("c", "Música notada (impresa)"),
        ("d", "Música notada manuscrita"), ("e", "Material cartográfico"),
        ("f", "Material cartográfico manuscrito"), ("g", "Medio proyectable (vídeo, película)"),
        ("i", "Grabación sonora no musical"), ("j", "Grabación sonora musical"),
        ("k", "Gráfico bidimensional no proyectable"), ("m", "Recurso electrónico"),
        ("o", "Kit"), ("p", "Materiales mixtos"), ("r", "Objeto tridimensional"),
        ("t", "Material textual manuscrito"),
    ],
    "cod_leader07_nivel_bibliografico": [
        ("a", "Parte componente de monografía"), ("b", "Parte componente de publicación seriada"),
        ("c", "Colección"), ("d", "Subunidad"), ("i", "Recurso integrable"),
        ("m", "Monografía / ítem"), ("s", "Publicación seriada"),
    ],
    "cod_tipo_fecha_008": [  # 008/06: determina el significado de fecha1 y fecha2
        ("b", "Sin fechas; fecha a. C."),
        ("c", "Recurso continuo en curso (fecha1 = inicio; fecha2 = 9999)"),
        ("d", "Recurso continuo cesado (fecha1 = inicio; fecha2 = fin)"),
        ("e", "Fecha detallada (fecha1 = año; fecha2 = mes y día)"),
        ("i", "Fechas extremas de una colección"),
        ("k", "Fechas de la mayor parte de una colección"),
        ("m", "Fechas múltiples (fecha1 = inicio; fecha2 = fin)"),
        ("n", "Fechas desconocidas"),
        ("p", "Fecha de distribución y fecha de producción/grabación"),
        ("q", "Fecha dudosa (fecha1 = más temprana; fecha2 = más tardía)"),
        ("r", "Reimpresión/reedición (fecha1 = reproducción; fecha2 = fecha de la original)"),
        ("s", "Fecha única conocida o probable (fecha2 en blanco)"),
        ("t", "Fecha de publicación (fecha1) y de copyright (fecha2)"),
        ("u", "Recurso continuo, estado desconocido"),
        ("|", "No codificado"),
    ],
    "cod_publico_008": [  # 008/22, solo libros
        ("#", "Desconocido o no especificado"), ("a", "Preescolar"), ("b", "Primaria"),
        ("c", "Preadolescente"), ("d", "Adolescente"), ("e", "Adulto"),
        ("f", "Especializado"), ("g", "General"), ("j", "Juvenil"), ("|", "No codificado"),
    ],
    "cod_forma_literaria_008": [  # 008/33, solo libros
        ("0", "No ficción"), ("1", "Ficción (sin especificar)"), ("c", "Cómics / historietas"),
        ("d", "Teatro"), ("e", "Ensayo"), ("f", "Novela"), ("h", "Humor, sátira"),
        ("i", "Cartas"), ("j", "Relatos cortos"), ("m", "Formas mixtas"), ("p", "Poesía"),
        ("s", "Discursos"), ("u", "Desconocida"), ("|", "No codificado"),
    ],
    "cod_traduccion_041_ind1": [
        ("#", "Sin información"), ("0", "No es ni incluye una traducción"),
        ("1", "Es o incluye una traducción"),
    ],
    "cod_tipo_publico_521_ind1": [
        ("#", "Público destinatario"), ("0", "Nivel de lectura (curso)"),
        ("1", "Edad de interés"), ("2", "Nivel de interés (curso)"),
        ("3", "Características especiales del público"), ("4", "Nivel de motivación/interés"),
        ("8", "Sin visualización asociada"),
    ],
}

# ---------------------------------------------------------------------------
# 2. CDU (clase 82). verificado = "si" solo si se ha comprobado en una fuente.
# ---------------------------------------------------------------------------
F_NLA = "Tabla de subdivisiones de la clase 82 (subject.nla.am), fuente secundaria"
F_BNE = "Registros BNE: 080 + 650 coincidentes (literarybibliography.eu / cervantesvirtual.com)"

CDU_FORMA = [
    # codigo, descripcion,                         verificado
    ("1",  "Poesía",                               "CDU"),
    ("2",  "Teatro",                               "CDU"),
    ("3",  "Prosa narrativa",                      "CDU"),
    ("31", "Novela",                               "CDU"),
    ("32", "Relato corto",                         "CDU"),
    ("34", "Cuentos populares",                    "fuente secundaria"),
    ("4",  "Ensayo",                               "CDU"),
    ("5",  "Oratoria, discursos",                  "CDU"),
    ("6",  "Cartas",                               "CDU"),
    ("7",  "Sátira y humor",                       "CDU"),
    ("8",  "Misceláneas",                          "CDU"),
    ("82", "Antologías",                           "fuente secundaria"),
    ("83", "Misceláneas (82-8)",                   "heredado del padre"),
    ("85", "Misceláneas (82-8)",                   "heredado del padre"),
    ("9",  "Otras formas literarias",              "CDU"),
    ("90", "Otras formas literarias (82-9)",       "heredado del padre"),
    ("91", "Cómic",                                "uso local verificado con títulos"),
    ("92", "Periodismo, literatura periódica",     "CDU"),
    ("93", "Literatura infantil",                  "fuente secundaria"),
    ("94", "Memorias, biografía, historiografía",  "CDU"),
    ("95", "Otras formas literarias (82-9)",       "heredado del padre"),
    ("96", "Otras formas literarias (82-9)",       "heredado del padre"),
    ("97", "Otras formas literarias (82-9)",       "heredado del padre"),
    ("98", "Otras formas literarias (82-9)",       "heredado del padre"),
    ("99", "Otras formas literarias (82-9)",       "heredado del padre"),
    ("0",  "Artefacto de extracción (no es forma)", "error v4"),
]

CDU_LENGUA = [  # código = lo que extrae el script en lengua_literatura_cdu (821.xxx)
    ("134.2", "Literatura en español / castellano", "si", F_BNE),
    ("134.1", "Literatura catalana", "si", F_BNE),
    ("134.4", "Literatura gallega", "si", F_BNE),
    ("361", "Literatura vasca", "si", F_BNE),
    ("111", "Literatura en inglés (pendiente)", "no", ""),
    ("133.1", "Literatura francesa (pendiente)", "no", ""),
    ("131.1", "Literatura italiana (pendiente)", "no", ""),
    ("134.3", "Literatura portuguesa (pendiente)", "no", ""),
    ("112.2", "Literatura alemana (pendiente)", "no", ""),
    ("161.1", "Literatura rusa (pendiente)", "no", ""),
]

CDU_LUGAR = [  # lugar_literatura_cdu, combinado con la lengua
    ("7/8", "América (con 134.2: literatura hispanoamericana)", "si", F_BNE),
]

# ---------------------------------------------------------------------------
# 3. LISTAS OFICIALES DE PAÍSES Y LENGUAS (loc.gov, XML)
# ---------------------------------------------------------------------------
URL_PAISES = "https://www.loc.gov/standards/codelists/countries.xml"
URL_LENGUAS = "https://www.loc.gov/standards/codelists/languages.xml"
NS = "{info:lc/xmlns/codelist-v1}"   # namespace de las listas de la LoC


def leer_xml_lista(url):
    """Usa el XML local si existe; si no, lo descarga identificándose como navegador
    (loc.gov devuelve 403 al agente por defecto de urllib)."""
    local = CARPETA_XML_LOC / url.rsplit("/", 1)[-1]
    if local.exists():
        print(f"  (leyendo copia local {local})")
        return local.read_bytes()
    peticion = urllib.request.Request(url, headers={
        "User-Agent": "Mozilla/5.0 (compatible; proyecto-academico/1.0)"})
    with urllib.request.urlopen(peticion, timeout=60) as r:
        return r.read()


def descargar_lista(url, etiqueta, con_region):
    """
    Descarga una lista MARC de la LoC y devuelve filas (codigo, nombre_en, [region], obsoleto).
    Cada <country>/<language> tiene un <name> hijo directo y uno o varios <code>;
    los códigos antiguos llevan status="obsolete". Se guardan todos: un registro
    antiguo puede usar un código obsoleto y debe poder traducirse.
    """
    raiz = ET.fromstring(leer_xml_lista(url))
    filas = []
    for elem in raiz.iter(NS + etiqueta):
        nombre_el = elem.find(NS + "name")
        nombre = (nombre_el.text or "").strip() if nombre_el is not None else ""
        region = ""
        if con_region:
            reg_el = elem.find(NS + "region")
            region = (reg_el.text or "").strip() if reg_el is not None else ""
        for code_el in elem.findall(NS + "code"):   # solo hijos directos
            codigo = (code_el.text or "").strip()
            obsoleto = "si" if code_el.get("status") == "obsolete" else "no"
            fila = [codigo, nombre] + ([region] if con_region else []) + [obsoleto]
            filas.append(fila)
    return filas


def escribir(nombre, cabecera, filas):
    ruta = CARPETA_SALIDA / f"{nombre}.csv"
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(cabecera)
        w.writerows(filas)
    print(f"  {ruta.name:40s} {len(filas):>5} filas")


def tablas_soporte_catalogo():
    CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
    print("Tablas del estándar MARC 21:")
    for nombre, filas in TABLAS_FIJAS.items():
        escribir(nombre, ["codigo", "descripcion", "fuente"], [(c, d, LOC) for c, d in filas])

    print("Tablas de la CDU:")
    cab = ["codigo", "descripcion", "verificado", "fuente"]
    escribir("cod_cdu_forma_literaria", cab, CDU_FORMA)
    escribir("cod_cdu_lengua_literatura", cab, CDU_LENGUA)
    escribir("cod_cdu_lugar_literatura", cab, CDU_LUGAR)

    print("Listas oficiales de la LoC:")
    for nombre, url, etiqueta, con_region, cabecera in [
        ("cod_pais_marc", URL_PAISES, "country", True, ["codigo", "nombre_en", "region", "obsoleto"]),
        ("cod_lengua_marc", URL_LENGUAS, "language", False, ["codigo", "nombre_en", "obsoleto"]),
    ]:
        try:
            escribir(nombre, cabecera, descargar_lista(url, etiqueta, con_region))
        except Exception as e:   # sin red o URL caída: el resto de tablas ya está escrito
            print(f"  AVISO: no se pudo obtener {url} ({e}).\n"
                  f"         Ábrela en el navegador, guárdala en {CARPETA_XML_LOC} y vuelve a ejecutar.")

tablas_soporte_catalogo()

# Guardar en DRIVE

In [ ]:
DRIVE_DESTINO = Path('/content/drive/')
DRIVE_DESTINO.mkdir(parents=True, exist_ok=True)

for f in ORIGEN.glob("*.csv"):
    shutil.copy(f, DRIVE_DESTINO / f.name)
    print("Copiado:", f.name)

# Fuerza copia a drive y desconecta
drive.flush_and_unmount()